In [1]:
import geopandas as gpd
import rasterio

In [2]:
datadir = '/projects/standard/lenkne/oboiko/EJ/'
raster_filepath = datadir + 'population/Dasymetric_Population_CONUS_2010_V3.tif'
aoi_counties_filepath = datadir + 'aoi_county_boundaries.gpkg'
wbdhuc12_filepath = datadir + 'WBD_National_GDB.gdb'
out_filepath = datadir + 'aoi_huc12_boundaries.gpkg'

In [3]:
# use the CRS of the gridded dataset (EPA population layer)
target_crs = rasterio.open(raster_filepath).crs
print(target_crs)

PROJCS["Albers_Conical_Equal_Area",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


In [4]:
# initial study area - 126 counties that border Mississippi River, previously assigned to a Qualitative Region
aoi_counties = gpd.read_file(aoi_counties_filepath)
# aggregate to qualitative regions
aoi_regions = aoi_counties[['Region', 'geometry']].dissolve(by='Region').reset_index()
aoi_regions

,Region,geometry
0,Chickasaw,"POLYGON ((535025.565 -219729.466, 534957.219 -..."
1,Confluence,"POLYGON ((583673.768 -91936.408, 583670.728 -9..."
2,Delta,"POLYGON ((466352.998 -589378.681, 466292.169 -..."
3,Driftless,"POLYGON ((396336.124 586488.875, 395733.219 58..."
4,Gorge,"POLYGON ((210826.373 818588.123, 210801.642 81..."
5,Gulf South,"MULTIPOLYGON (((544754.442 -927100.703, 544744..."
6,Headwaters,"POLYGON ((169464.691 968923.052, 169456.828 96..."
7,Lower Mississippi,"POLYGON ((448216.41 -753718.292, 447966.537 -7..."
8,Working River,"POLYGON ((405341.844 193906.689, 404600.737 19..."


In [5]:
%%time
# read the national HUC12 data
wbdhuc12 = gpd.read_file(wbdhuc12_filepath, layer='WBDHU12', METHOD='SKIP')

/users/2/oboiko/.conda/envs/geo/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver OpenFileGDB does not support open option METHOD
  return ogr_read(
/users/2/oboiko/.conda/envs/geo/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts.  The processing may be really slow.  You can skip the processing by setting METHOD=SKIP.
  return ogr_read(


CPU times: user 29.7 s, sys: 4.1 s, total: 33.8 s
Wall time: 58.1 s


In [6]:
%%time
# select WBD HUC12 that interect counties
wbdhuc12_selection = gpd.sjoin(wbdhuc12, aoi_regions.to_crs(wbdhuc12.crs), predicate='intersects')

CPU times: user 3.58 s, sys: 13.1 ms, total: 3.6 s
Wall time: 3.66 s


In [7]:
# calculate intersection area for each combination of HUC12 and Region
aoi_regions = aoi_regions.to_crs(target_crs)
wbdhuc12_selection = wbdhuc12_selection.to_crs(target_crs)
wbdhuc12_selection['intersection_area'] = wbdhuc12_selection.apply(
    lambda x: sum(aoi_regions[aoi_regions['Region']==x.Region].geometry.intersection(x.geometry).area), axis=1
)
# filter out duplicates - keep QR region assignment by highest captured area
final_selection = wbdhuc12_selection.sort_values('intersection_area', ascending=False).drop_duplicates('huc12').sort_index()
print (f'Got {len(final_selection)} rows')
# keep only some columns
cols = ['huc12', 'name', 'areasqkm', 'states', 'geometry', 'Region']
final_selection = final_selection[cols].copy()
# preview resulting GeoDataFrame
final_selection.head()

Got 2576 rows


,huc12,name,areasqkm,states,geometry,Region
27,070300050401,Forest Lake-Sunrise River,43.43,MN,"MULTIPOLYGON (((240228.337 2483439.303, 240271...",Gorge
28,070900070402,Fairfield Ditch Number 1-Green River,90.16,IL,"MULTIPOLYGON (((517568.047 2070542.219, 517539...",Working River
62,070300030103,West Branch Kettle River,101.59,MN,"MULTIPOLYGON (((234614.153 2633575.283, 234693...",Headwaters
71,070400080902,Crystal Creek,41.77,MN,"MULTIPOLYGON (((362547.97 2317024.238, 362554....",Driftless
78,070400030605,Rose Valley,39.09,WI,"MULTIPOLYGON (((332805.751 2374160.973, 332815...",Driftless


In [14]:
# save to a file
# final_selection.to_file(out_filepath)